# pgembed: embedded Postgres logical replication

This notebook starts two embedded PostgreSQL clusters from `pgembed`: one primary and one secondary. The primary is restarted once after `ALTER SYSTEM SET wal_level = 'logical'`, then it publishes a table and the secondary subscribes to that publication over Unix domain sockets. Data is written to the primary and read from the secondary.


## Environment

Install the optional Postgres dependencies from `pyproject.toml` before running this notebook:

```bash
uv sync --extra postgres
uv run jupyter notebook pgembed_logical_replication.ipynb
```


In [1]:
import atexit
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

import pgembed
from pgembed._commands import POSTGRES_BIN_PATH

BASE_DIR = Path(tempfile.mkdtemp(prefix="pgembed-logical-replication-"))
PRIMARY_DIR = BASE_DIR / "primary"
SECONDARY_DIR = BASE_DIR / "secondary"

BASE_DIR


PosixPath('/var/folders/mg/8cz953zd56g5kn6ls6xpy7t80000gn/T/pgembed-logical-replication-t1yvmw23')

## Helpers

The example follows pgembed's normal `get_server()` / `get_uri()` pattern. On macOS and Linux, pgembed starts Postgres on Unix domain sockets, so no TCP ports or `pg_hba.conf` edits are needed. The only special handling is restarting the primary after `ALTER SYSTEM SET wal_level = 'logical'`, because `wal_level` is read at server start.


In [2]:
PSQL = Path(POSTGRES_BIN_PATH) / "psql"


def run(cmd, **kwargs):
    with tempfile.TemporaryFile("w+") as stdout, tempfile.TemporaryFile("w+") as stderr:
        subprocess.run(
            [str(part) for part in cmd],
            check=True,
            text=True,
            stdout=stdout,
            stderr=stderr,
            **kwargs,
        )
        stdout.seek(0)
        return stdout.read().strip()


def sql(uri, statement, *, quiet=False):
    args = [
        PSQL,
        uri,
        "-v",
        "ON_ERROR_STOP=1",
        "-X",
    ]
    if quiet:
        args.append("-q")
    args.extend(["-c", statement])
    return run(args)


def cleanup():
    for server in (globals().get("secondary"), globals().get("primary")):
        if server is not None:
            server.cleanup()
    shutil.rmtree(BASE_DIR, ignore_errors=True)


atexit.register(cleanup)


<function __main__.cleanup()>

## Start primary and secondary

`pgembed.get_server()` initializes and starts each cluster. The primary is restarted after `ALTER SYSTEM SET wal_level = 'logical'` so logical WAL is active before the publication is created.


In [3]:
primary = pgembed.get_server(PRIMARY_DIR)
primary_uri = primary.get_uri("postgres")

sql(primary_uri, "ALTER SYSTEM SET wal_level = 'logical';", quiet=True)
primary.cleanup()
primary = pgembed.get_server(PRIMARY_DIR)
primary_uri = primary.get_uri("postgres")

secondary = pgembed.get_server(SECONDARY_DIR)
secondary_uri = secondary.get_uri("postgres")

{
    "primary": primary_uri,
    "secondary": secondary_uri,
}


{'primary': 'postgresql://postgres:@/postgres?host=/Users/arun/Library/Caches/TemporaryItems/python_PostgresServer/4f65bf3cbb',
 'secondary': 'postgresql://postgres:@/postgres?host=/Users/arun/Library/Caches/TemporaryItems/python_PostgresServer/4ceac449a2'}

In [4]:
sql(primary_uri, "SHOW wal_level;")


'wal_level \n-----------\n logical\n(1 row)'

## Create the publication

Logical replication sends row changes through WAL. DDL is not replicated, so the table schema is created on both clusters before the subscription is created.


In [5]:
schema_sql = """
DROP TABLE IF EXISTS replicated_notes;
CREATE TABLE replicated_notes (
    id integer PRIMARY KEY,
    body text NOT NULL,
    written_at timestamptz NOT NULL DEFAULT now()
);
"""

sql(primary_uri, schema_sql, quiet=True)
sql(secondary_uri, schema_sql, quiet=True)

sql(
    primary_uri,
    """
    INSERT INTO replicated_notes (id, body)
    VALUES (1, 'seed row before subscription');

    DROP PUBLICATION IF EXISTS demo_pub;
    CREATE PUBLICATION demo_pub FOR TABLE replicated_notes;
    """,
    quiet=True,
)

sql(primary_uri, "SELECT * FROM pg_publication;")


'oid  | pubname  | pubowner | puballtables | pubinsert | pubupdate | pubdelete | pubtruncate | pubviaroot \n-------+----------+----------+--------------+-----------+-----------+-----------+-------------+------------\n 16392 | demo_pub |       10 | f            | t         | t         | t         | t           | f\n(1 row)'

## Subscribe the secondary

`copy_data = true` copies the seed row, then the subscription keeps applying new primary-side changes from logical WAL.


In [6]:
sql(secondary_uri, "DROP SUBSCRIPTION IF EXISTS demo_sub;", quiet=True)
sql(
    secondary_uri,
    f"""
    CREATE SUBSCRIPTION demo_sub
    CONNECTION '{primary_uri}'
    PUBLICATION demo_pub
    WITH (copy_data = true);
    """,
    quiet=True,
)

sql(secondary_uri, "SELECT subname, subenabled FROM pg_subscription;")


'subname  | subenabled \n----------+------------\n demo_sub | t\n(1 row)'

## Write to primary, read from secondary

The polling loop waits until the subscriber has applied the copied seed row plus two new rows written on the primary.


In [7]:
sql(
    primary_uri,
    """
    INSERT INTO replicated_notes (id, body)
    VALUES
        (2, 'first row written on the primary'),
        (3, 'second row written on the primary')
    ON CONFLICT (id) DO UPDATE SET body = EXCLUDED.body;
    """,
    quiet=True,
)

deadline = time.time() + 30
while time.time() < deadline:
    count_text = sql(
        secondary_uri,
        "SELECT count(*) FROM replicated_notes;",
        quiet=True,
    )
    if "3" in count_text.split():
        break
    time.sleep(0.25)
else:
    raise TimeoutError("secondary did not receive all replicated rows")

print(sql(secondary_uri, "SELECT id, body FROM replicated_notes ORDER BY id;"))


id |               body                
----+-----------------------------------
  1 | seed row before subscription
  2 | first row written on the primary
  3 | second row written on the primary
(3 rows)


## Replication status

The primary exposes the logical replication sender, and the secondary exposes subscription worker state.


In [8]:
print("Primary replication sender:")
print(sql(primary_uri, "SELECT application_name, state, sync_state FROM pg_stat_replication;"))

print()
print("Secondary subscription state:")
print(sql(secondary_uri, "SELECT subname, received_lsn, latest_end_lsn FROM pg_stat_subscription;"))


Primary replication sender:
application_name |   state   | sync_state 
------------------+-----------+------------
 demo_sub         | streaming | async
(1 row)

Secondary subscription state:
subname  | received_lsn | latest_end_lsn 
----------+--------------+----------------
 demo_sub | 0/14C6418    | 0/14C6418
(1 row)


## Cleanup

Run this cell when you are done. It drops the embedded clusters and removes the temporary data directories.


In [9]:
cleanup()
atexit.unregister(cleanup)
